# PDA

In [1]:
# Libraries importing
from pyspark import keyword_only
from pyspark.ml import Transformer
from pyspark.ml.param.shared import HasInputCol, HasOutputCol, Param, Params, TypeConverters
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType
import pyspark.sql.functions as F
import math
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType, DateType, ArrayType, FloatType

In [2]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

In [235]:
from pyspark.sql import SparkSession

# Add here your team number teamx
team = 4

# location of your Hive database in HDFS
warehouse = "/user/team4/project/hive/warehouse"
# .config("spark.executor.instances", "2") \

spark = SparkSession.builder \
        .appName("{} - spark ML".format(team)) \
        .master("yarn") \
        .config("spark.executor.cores", 3) \
        .config("spark.executor.memory", "12g") \
        .config("spark.driver.memory", "2g") \
        .config("spark.submit.deployMode", "client")\
        .config("hive.metastore.uris", "thrift://hadoop-02.uni.innopolis.ru:9883") \
        .config("spark.sql.warehouse.dir", warehouse) \
        .config("spark.sql.avro.compression.codec", "snappy") \
        .enableHiveSupport() \
        .getOrCreate()

In [5]:
spark.sql("SHOW DATABASES").show()

+--------------------+
|           namespace|
+--------------------+
|             default|
|           ml_stage3|
|             retake1|
|             root_db|
|                show|
|    team00_projectdb|
|          team0_dbms|
|     team0_projectdb|
|             team0db|
|            team0db1|
|            team0db2|
|    team11_projectdb|
|           team12_db|
|team12_hive_proje...|
|    team12_projectdb|
|    team13_projectdb|
|team13_projectdb_...|
|    team14_projectdb|
|    team15_projectdb|
|    team16_projectdb|
+--------------------+
only showing top 20 rows



In [6]:
spark.sql("USE team4_projectdb")
spark.sql("SHOW TABLES").show()

+---------------+------------------+-----------+
|      namespace|         tableName|isTemporary|
+---------------+------------------+-----------+
|team4_projectdb|           flights|      false|
|team4_projectdb|   hyperparameters|      false|
|team4_projectdb|      optimization|      false|
|team4_projectdb|prediction_samples|      false|
|team4_projectdb|        q1_results|      false|
|team4_projectdb|        q2_results|      false|
|team4_projectdb|        q3_results|      false|
|team4_projectdb|        q4_results|      false|
|team4_projectdb|        q5_results|      false|
|team4_projectdb|        q6_results|      false|
+---------------+------------------+-----------+



## Data loading

In [241]:
# Load dataset
df = spark.sql("SELECT * FROM team4_projectdb.flights")

In [8]:
# Shape
rows = df.count()
cols = len(df.columns)
print(f'Rows: {rows}\nColumns: {cols}')

Rows: 6311871
Columns: 61


In [9]:
# Schema
df.printSchema()

root
 |-- flightdate: date (nullable = true)
 |-- airline: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- cancelled: boolean (nullable = true)
 |-- diverted: boolean (nullable = true)
 |-- crsdeptime: timestamp (nullable = true)
 |-- deptime: timestamp (nullable = true)
 |-- depdelayminutes: integer (nullable = true)
 |-- depdelay: integer (nullable = true)
 |-- arrtime: timestamp (nullable = true)
 |-- arrdelayminutes: integer (nullable = true)
 |-- airtime: integer (nullable = true)
 |-- crselapsedtime: integer (nullable = true)
 |-- actualelapsedtime: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- dayofmonth: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)
 |-- marketing_airline_network: string (nullable = true)
 |-- operated_or_branded_code_share_partners: string (nullable = true)
 |-- dot_id_marketing_air

In [10]:
# 1 sample
df.show(1)

+----------+--------------------+------+----+---------+--------+-------------------+-------------------+---------------+--------+-------------------+---------------+-------+--------------+-----------------+--------+----+-------+----------+---------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+--------------+-----------+---------------+--------------------+---------+-------------+----------------+----------------+------------+---------+-------------+--------------------+-------+--------+--------------------+----------+-------+-------------------+-------------------+------+-------------------+--------+--------+------------------+----------+-------------+------------------+-----+
|flightdate|             airline|origin

The original dataset contains 6,311,871 rows and 61 columns, some of which have null values. Since nulls are contained in almost all rows for which the label value is True, it was decided not to drop such rows, but to perform custom imputing.

In [242]:
# Dropped rows
from pyspark.sql.functions import coalesce

def fillna_mean(df, include=set()): 
    means = df.agg(*(
        F.mean(x).alias(x) for x in df.columns if x in include
    ))
    return df.fillna(means.first().asDict())

def fillna_mode(df, column): 
    moda = df.groupby(column).count()\
                             .orderBy('count', ascending=False)\
                             .select(column)\
                             .collect()[1][column]
    return df.na.fill(value = moda, subset = [column])

# Small number of rows containing nulls
df = df.dropna(subset = ["crselapsedtime", 'divairportlandings'])

# Fill DepTime nulls
df = df.withColumn("deptime", coalesce(df.deptime,
                                         df.crsdeptime))
# Fill ArrTime nulls
df = df.withColumn("arrtime", coalesce(df.arrtime,
                                         df.crsarrtime))
 # Fill ActualElapsedTime nulls
df = df.withColumn("actualelapsedtime", coalesce(df.actualelapsedtime, 
                                                   df.crselapsedtime))
# Fill null values with mean
df = fillna_mean(df, ['taxiout', 'wheelsoff', 'wheelson', 'taxiin'])
# Fill null values with mode
df = fillna_mode(df, 'tail_number')

# Fill AirTime null values based on airtime formula
df = df.withColumn('tmp', (df['actualelapsedtime'] - df['taxiout'] - df['taxiin']))
df = df.withColumn("airtime", coalesce(df.airtime, 
                                         df.tmp)).drop('tmp')

# Fill with zeroes and False
df = df.na.fill(value = 0, subset = ["depdelayminutes", 'depdelay', 
                                       "arrdelayminutes", 'arrdelay',
                                       'arrivaldelaygroups', 
                                       'departuredelaygroups'])
df = df.na.fill(value = False, subset = ['depdel15', 'arrdel15'])

## Feature extraction

In [243]:
# Features' categories
numerical = ['depdelayminutes', 'depdelay', 'arrdelayminutes', 'airtime',
             'crselapsedtime', 'actualelapsedtime', 'distance', 'year', 
              'flight_number_marketing_airline', 'flight_number_operating_airline', 
             'originairportid', 'originairportseqid', 'origincitymarketid', 
             'originstatefips', 'originwac', 'destairportid', 'destairportseqid', 
             'destcitymarketid', 'deststatefips', 'destwac',  
             'departuredelaygroups', 'taxiout', 'taxiin', 'arrdelay', 
              'arrivaldelaygroups', 'distancegroup']

cyclical = ['quarter', 'month', 'dayofmonth', 'dayofweek']
time_features = ['crsdeptime', 'deptime', 'crsarrtime', 'arrtime',  'wheelson', 'wheelsoff']
date_features = ['flightdate']
boolean_features = ['diverted', 'depdel15', 'arrdel15']

categorical_as_cont = ['origin', 'dest', 'tail_number', 'origincityname', 
                       'destcityname']
categorical_ohe = ['airline', 'marketing_airline_network', 
                   'operated_or_branded_code_share_partners', 
                   'iata_code_marketing_airline', 'operating_airline', 
                   'iata_code_operating_airline', 'originstate', 
                   'originstatename', 'deststate', 'deststatename', 
                   'deptimeblk', 'arrtimeblk', 
                   'divairportlandings', 'dot_id_marketing_airline','dot_id_operating_airline']

label = 'cancelled'

print(f"Categorical features: {', '.join(categorical_ohe)}\n" + 
      f"Categorical features with many uniques: {', '.join(categorical_as_cont)}\n" + 
      f"Numerical features: {', '.join(numerical)}\n" + 
      f"Time features: {', '.join(time_features)}\n" + 
      f"Boolean features: {', '.join(boolean_features)}\n" + 
      f"Target: {label}")

Categorical features: airline, marketing_airline_network, operated_or_branded_code_share_partners, iata_code_marketing_airline, operating_airline, iata_code_operating_airline, originstate, originstatename, deststate, deststatename, deptimeblk, arrtimeblk, divairportlandings, dot_id_marketing_airline, dot_id_operating_airline
Categorical features with many uniques: origin, dest, tail_number, origincityname, destcityname
Numerical features: depdelayminutes, depdelay, arrdelayminutes, airtime, crselapsedtime, actualelapsedtime, distance, year, flight_number_marketing_airline, flight_number_operating_airline, originairportid, originairportseqid, origincitymarketid, originstatefips, originwac, destairportid, destairportseqid, destcitymarketid, deststatefips, destwac, departuredelaygroups, taxiout, taxiin, arrdelay, arrivaldelaygroups, distancegroup
Time features: crsdeptime, deptime, crsarrtime, arrtime, wheelson, wheelsoff
Boolean features: diverted, depdel15, arrdel15
Target: cancelled


After the stage of nulls elimination, one need to do proper sampling for dataset, as it is very imbalanced in term of label columns 'cancelled'. The reason for that is simply that dataset contains less records about cancelled flights, than successfully completed.

In [244]:
# Sample data as it's really imbalansed wrt label
major_df = df.filter(F.col(label) == 'false')
minor_df = df.filter(F.col(label) == 'true')

result_frac = minor_df.count()/major_df.count()

df_sampled = df.sampleBy(label, fractions={False: result_frac, True: 1}, seed=123)

A common method for encoding different cyclical data, including date and time, is to transform this data into two dimensions using a sine and cosine transformations. 

In [14]:
# Custom transformer to encode cyclical features with sin and cos
class TimeTransformer(Transformer, HasInputCol, HasOutputCol, DefaultParamsReadable, DefaultParamsWritable):
    input_col = Param(Params._dummy(), "input_col", "input column name.", typeConverter=TypeConverters.toString)
    output_col = Param(Params._dummy(), "output_col", "output column name.", typeConverter=TypeConverters.toString)

    @keyword_only
    def __init__(self, input_col: str = "input", output_col: str = "output"):
        super(TimeTransformer, self).__init__()
        self._setDefault(input_col=None, output_col=None)
        kwargs = self._input_kwargs
        self.set_params(**kwargs)
        self.set_coef()

    def set_coef(self, coef: float = 12.0):
        self.coef = coef
        return self

    @keyword_only
    def set_params(self, input_col: str = "input", output_col: str = "output"):
        kwargs = self._input_kwargs
        self._set(**kwargs)

    def get_input_col(self):
        return self.getOrDefault(self.input_col)

    def get_output_col(self):
        return self.getOrDefault(self.output_col)

    def _transform(self, df: DataFrame):
        input_col = self.get_input_col()
        output_col = self.get_output_col()
        output_col_cos = output_col + "_cos"
        output_col_sin = output_col + "_sin"

        transform_udf = F.udf(lambda x: str(2*math.pi*int(x)/self.coef), StringType())
        return df.withColumn(output_col_cos, F.cos(transform_udf(input_col)))\
                 .withColumn(output_col_sin, F.sin(transform_udf(input_col)))

The main pipeline for features extraction includes indexers, time transformer, one-hot encoder and minmax scaler.

In [245]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler, MinMaxScaler
from pyspark.ml import Pipeline

# Convert boolean
for b in boolean_features + [label]:
    df_sampled = df_sampled.withColumn(b, F.col(b).cast('integer'))

# String indexers for categorical features
indexer = StringIndexer()
indexer.setHandleInvalid("keep")
indexer.setInputCols(categorical_ohe)
features_idx = list(map(lambda x : x + "_idx", categorical_ohe))
indexer.setOutputCols(features_idx)

cat_indexer = StringIndexer()
cat_indexer.setHandleInvalid("keep")
cat_indexer.setInputCols(categorical_as_cont)
features_cat_idx = list(map(lambda x : x + "_idx", categorical_as_cont))
cat_indexer.setOutputCols(features_cat_idx)

# One-hot encoding for categorical
encoders = []
for f in features_idx:
    encoders.append(OneHotEncoder(inputCol = f, outputCol = f + "_enc"))

# Encode parsed cyclical
period = {
    "quarter": 4, 
    "month": 12, 
    "dayofmonth": 31, 
    "dayofweek": 7
}
cyclical_encoders = []
for c in cyclical:
    cyclical_encoders.append(TimeTransformer(input_col = c, output_col = c).set_coef(coef = period[c]))
    
# Encode time
time_idx = []
time_encoders = []
for t in time_features:
    time_idx.append(t + '_hour')
    time_idx.append(t + '_minute')
    time_idx.append(t + '_day')
    time_idx.append(t + '_month')

    df_sampled = df_sampled.withColumn(t + '_hour', F.hour(t))
    df_sampled = df_sampled.withColumn(t + '_minute', F.minute(t))
    df_sampled = df_sampled.withColumn(t + '_day', F.dayofmonth(t))
    df_sampled = df_sampled.withColumn(t + '_month', F.month(t))
    
    time_encoders.append(TimeTransformer(input_col = t + '_hour', output_col = t + '_hour').set_coef(coef = 24.0))
    time_encoders.append(TimeTransformer(input_col = t + '_minute', output_col = t + '_minute').set_coef(coef = 60.0))
    time_encoders.append(TimeTransformer(input_col = t + '_day', output_col = t + '_day').set_coef(coef = 31.0))
    time_encoders.append(TimeTransformer(input_col = t + '_month', output_col = t + '_month').set_coef(coef = 12.0))

# Encode date
date_idx = []
date_encoders = []
for d in date_features:
    date_idx.append(d + '_day')
    date_idx.append(d + '_month')
    
    df_sampled = df_sampled.withColumn(d + '_day', F.dayofmonth(t))
    df_sampled = df_sampled.withColumn(d + '_month', F.month(t))
    
    date_encoders.append(TimeTransformer(input_col = d + '_day', 
                                         output_col = d + '_day').set_coef(coef = 31.0))
    date_encoders.append(TimeTransformer(input_col = d + '_month', 
                                         output_col = d + '_month').set_coef(coef = 12.0))


# Assemble all features
assembler = VectorAssembler(
    inputCols = [f + "_enc" for f in features_idx] + features_cat_idx + numerical + boolean_features + [f + "_sin" for f in time_idx] + \
                [f + "_cos" for f in time_idx] + [f + "_sin" for f in date_idx] + [f + "_cos" for f in date_idx] + [f + "_sin" for f in cyclical] + \
                [f + "_cos" for f in cyclical],
    outputCol='features_unscaled'
    )

# MinMax Scaler
scaler = MinMaxScaler(inputCol = "features_unscaled", outputCol = "features")

# Apply pipeline
pipeline = Pipeline(stages = [indexer, cat_indexer] + encoders + time_encoders + date_encoders + cyclical_encoders + [assembler, scaler])
features_pipeline_model = pipeline.fit(df_sampled)
df_enc = features_pipeline_model.transform(df_sampled)

# Indexing the labels
label_indexer = StringIndexer()
label_indexer.setInputCol(label)
label_indexer.setOutputCol('label')
label_idx_model = label_indexer.fit(df_enc)

# Apply the indexer
df_labeled = label_idx_model.transform(df_enc)

In [246]:
# Get features and label
df_proj = df_labeled.select('features', 'label')
df_proj.show(10)

+--------------------+-----+
|            features|label|
+--------------------+-----+
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
|(481,[9,28,40,53,...|  1.0|
+--------------------+-----+
only showing top 10 rows



In [17]:
# Number of new features
df_proj.select('features').collect()[0]['features'].toArray().shape

(481,)

## PCA preparation

In [18]:
# Reconstruct feature pipeline up to the unscaled features (without MinMaxScaler)
assembler_raw = VectorAssembler(
    inputCols=[f + "_enc" for f in features_idx] + features_cat_idx + numerical + boolean_features +
              [f + "_sin" for f in time_idx] + [f + "_cos" for f in time_idx] +
              [f + "_sin" for f in date_idx] + [f + "_cos" for f in date_idx] +
              [f + "_sin" for f in cyclical] + [f + "_cos" for f in cyclical],
    outputCol='features_raw'
)

pipeline_raw = Pipeline(stages=[indexer, cat_indexer] + encoders + time_encoders + date_encoders + cyclical_encoders + [assembler_raw])
raw_model = pipeline_raw.fit(df_sampled)
df_raw = raw_model.transform(df_sampled)
df_raw_labeled = label_idx_model.transform(df_raw)

train_raw, test_raw = df_raw_labeled.select('features_raw', 'label').randomSplit([0.7, 0.3], seed=42)

In [19]:
from pyspark.ml.feature import StandardScaler, PCA

# Scaling
scaler_pca = StandardScaler(inputCol="features_raw", outputCol="scaled_features",
                            withMean=True, withStd=False)
scaler_model = scaler_pca.fit(train_raw)
train_scaled = scaler_model.transform(train_raw)
test_scaled = scaler_model.transform(test_raw)

# Determine optimal k by cumulative explained variance (>95%)
n_features = train_scaled.select("scaled_features").first()[0].size
pca_full = PCA(k=n_features, inputCol="scaled_features", outputCol="pca_features")
pca_full_model = pca_full.fit(train_scaled)
explained_var = pca_full_model.explainedVariance.cumsum()
k_optimal = next(i + 1 for i, v in enumerate(explained_var) if v >= 0.95)
print(f"Optimal number of PCA components (95% variance): {k_optimal}")

# PCA with selected k
pca = PCA(k=k_optimal, inputCol="scaled_features", outputCol="pca_features")
pca_model = pca.fit(train_scaled)

train_pca = pca_model.transform(train_scaled).select("pca_features", "label")
test_pca = pca_model.transform(test_scaled).select("pca_features", "label")

# Rename to 'features' for consistency
train_pca = train_pca.withColumnRenamed("pca_features", "features")
test_pca = test_pca.withColumnRenamed("pca_features", "features")

Optimal number of PCA components (95% variance): 2


In [21]:
train_pca.select('features').collect()[0]['features'].toArray().shape

(2,)

## Modeling

In [247]:
# Train/Test split
trainRatio = 0.7

train_df, test_df = df_proj.randomSplit([trainRatio, 1 - trainRatio], seed = 42)
print(f"Ratio: {trainRatio}\nTrain size: {train_df.count()}\nTest size: {test_df.count()}")

Ratio: 0.7
Train size: 155159
Test size: 66236


In [22]:
# A function to run commands
import os
def run(command):
    return os.popen(command).read()

# Save to HDFS
train_df.select("features", "label")\
    .write\
    .mode("overwrite")\
    .format('json')\
    .save("/user/team4/project/data/train")

# Add localy
run("hdfs dfs -cat /user/team4/project/data/train/*.json > ~/project/bigdata-project/data/train.json")

# Save to HDFS
test_df.select("features", "label")\
    .write\
    .mode("overwrite")\
    .format("json")\
    .save("/user/team4/project/data/test")

# Add localy
run("hdfs dfs -cat /user/team4/project/data/test/*.json > ~/project/bigdata-project/data/test.json")

''

## Model 1

Random Forest is an ensemble learning method that combines multiple decision trees to create a more robust and accurate model. It builds multiple decision trees to improve the overall performance using a subset of the training data.

In [23]:
# Train first model
from pyspark.ml.classification import RandomForestClassifier

rf_calssifier = RandomForestClassifier()
rf_model = rf_calssifier.fit(train_df)

In [24]:
# Test first model
rf_predictions = rf_model.transform(test_df)
rf_predictions.show(10)

+--------------------+-----+--------------------+--------------------+----------+
|            features|label|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(481,[0,24,32,49,...|  0.0|[19.7298264124152...|[0.98649132062076...|       0.0|
|(481,[0,24,32,49,...|  0.0|[19.7325420497365...|[0.98662710248682...|       0.0|
|(481,[0,24,32,49,...|  1.0|[0.03879273395665...|[0.00193963669783...|       1.0|
|(481,[0,24,32,49,...|  1.0|[0.06750616825466...|[0.00337530841273...|       1.0|
|(481,[0,24,32,49,...|  1.0|[0.10936034581361...|[0.00546801729068...|       1.0|
|(481,[0,24,32,49,...|  1.0|[0.02373558455497...|[0.00118677922774...|       1.0|
|(481,[0,24,32,49,...|  0.0|[19.8754905907390...|[0.99377452953695...|       0.0|
|(481,[0,24,32,49,...|  0.0|[19.6972924703136...|[0.98486462351568...|       0.0|
|(481,[0,24,32,49,...|  1.0|[0.00939522702954...|[4.69761351477439...|       1.0|
|(481,[0,24,32,4

In [25]:
# Evaluation
from pyspark.ml.evaluation import BinaryClassificationEvaluator

rf_evaluator_roc = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderROC")

rf_roc = rf_evaluator_roc.evaluate(rf_predictions)

rf_evaluator_pr = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderPR")

rf_pr = rf_evaluator_pr.evaluate(rf_predictions)

print(f"Test area under ROC: {rf_roc}\nTest area under PR: {rf_pr}")

Test area under ROC: 0.9999100746380505
Test area under PR: 0.9998574216520281


In [26]:
# Hyper-parameter optimization
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print(f"Hyperparameters:\n\t maxDepth - {[5, 10]}\n\t impurity - {['entropy', 'gini']}\n\t numTrees - {[10, 20]}")

rf_grid = ParamGridBuilder()
rf_grid = rf_grid.addGrid(rf_model.maxDepth, [5, 10])\
                 .addGrid(rf_model.impurity, ['entropy', 'gini'])\
                 .addGrid(rf_model.numTrees, [10, 20])\
                 .build()

rf_cv = CrossValidator(estimator = rf_calssifier,
                    estimatorParamMaps = rf_grid,
                    evaluator = rf_evaluator_roc,
                    parallelism = 5,
                    numFolds=3)

rf_cvModel = rf_cv.fit(train_df)
model1 = rf_cvModel.bestModel

print(f"\nBest hyperparameters:\n\t maxDepth - {model1.getMaxDepth()}\n\t impurity - {model1.getImpurity()}\n\t numTrees - {model1.getNumTrees}\n")

model1

Hyperparameters:
	 maxDepth - [5, 10]
	 impurity - ['entropy', 'gini']
	 numTrees - [10, 20]

Best hyperparameters:
	 maxDepth - 10
	 impurity - gini
	 numTrees - 10



RandomForestClassificationModel: uid=RandomForestClassifier_652c75c2e8b0, numTrees=10, numClasses=2, numFeatures=481

In [27]:
model1.write().overwrite().save("/user/team4/project/models/model1")
run("hdfs dfs -get /user/team4/project/models/model1 ~/project/bigdata-project/models/model1")

''

In [28]:
rf_predictions_grid = model1.transform(test_df)
rf_predictions_grid.show(10)

+--------------------+-----+--------------------+--------------------+----------+
|            features|label|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(481,[0,24,32,49,...|  1.0|          [0.0,10.0]|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  0.0|[9.99913983427473...|[0.99991398342747...|       0.0|
|(481,[0,24,32,49,...|  1.0|          [0.0,10.0]|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  0.0|[9.99913983427473...|[0.99991398342747...|       0.0|
|(481,[0,24,32,49,...|  1.0|[0.00114111829593...|[1.14111829593001...|       1.0|
|(481,[0,24,32,49,...|  1.0|          [0.0,10.0]|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  0.0|          [10.0,0.0]|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  1.0|[0.00105042016806...|[1.05042016806722...|       1.0|
|(481,[0,24,32,49,...|  1.0|          [0.0,10.0]|           [0.0,1.0]|       1.0|
|(481,[0,24,32,4

In [29]:
rf_predictions_grid.select("label", "prediction")\
    .coalesce(1)\
    .write\
    .mode("overwrite")\
    .format("csv")\
    .option("sep", ",")\
    .option("header","true")\
    .save("/user/team4/project/output/model1_predictions")
run("hdfs dfs -cat /user/team4/project/output/model1_predictions/*.csv > ~/project/bigdata-project/output/model1_predictions.csv")

''

In [30]:
rf_evaluator_best_roc = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderROC")

rf_best_roc = rf_evaluator_best_roc.evaluate(rf_predictions_grid)

rf_evaluator_best_pr = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderPR")

rf_best_pr = rf_evaluator_best_pr.evaluate(rf_predictions_grid)

print(f"Test area under ROC for best model: {rf_best_roc}\nTest area under PR for best model: {rf_best_pr}")

Test area under ROC for best model: 0.9998649743451256
Test area under PR for best model: 0.9999550792426315


In [31]:
print(f"Test area under ROC for first model: {rf_roc}\nTest area under PR for first model: {rf_pr}")
print(f"Test area under ROC for best model: {rf_best_roc}\nTest area under PR for best model: {rf_best_pr}")
print(f"\nTest area under ROC increased by {rf_best_roc - rf_roc} with optimization\nTest area under PR increased " + 
      f"by {rf_best_pr - rf_pr} with optimization")

Test area under ROC for first model: 0.9999100746380505
Test area under PR for first model: 0.9998574216520281
Test area under ROC for best model: 0.9998649743451256
Test area under PR for best model: 0.9999550792426315

Test area under ROC increased by -4.510029292492668e-05 with optimization
Test area under PR increased by 9.765759060342827e-05 with optimization


### Random Forest on PCA

In [32]:
# RF on PCA
rf_pca = RandomForestClassifier(labelCol="label", featuresCol="features")
rf_pca_model = rf_pca.fit(train_pca)

rf_pca_pred = rf_pca_model.transform(test_pca)
rf_pca_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC").evaluate(rf_pca_pred)
rf_pca_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderPR").evaluate(rf_pca_pred)

print(f"Test area under ROC: {rf_pca_roc}\nTest area under PR: {rf_pca_pr}")

Test area under ROC: 0.5357261630905746
Test area under PR: 0.5404691212513495


In [33]:
# Tuning RF on PCA
rf_pca_grid = ParamGridBuilder() \
    .addGrid(rf_pca_model.maxDepth, [5, 10]) \
    .addGrid(rf_pca_model.impurity, ['entropy', 'gini']) \
    .addGrid(rf_pca_model.numTrees, [10, 20]) \
    .build()

rf_pca_cv = CrossValidator(estimator=RandomForestClassifier(labelCol="label", featuresCol="features"),
                           estimatorParamMaps=rf_pca_grid,
                           evaluator=BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC"),
                           parallelism=5, numFolds=3)

rf_pca_cvModel = rf_pca_cv.fit(train_pca)
best_rf_pca = rf_pca_cvModel.bestModel
print(f"\nBest hyperparameters:\n\t maxDepth - {best_rf_pca.getMaxDepth()}\n\t impurity - {best_rf_pca.getImpurity()}\n\t numTrees - {best_rf_pca.getNumTrees}\n")

best_rf_pca


Best hyperparameters:
	 maxDepth - 5
	 impurity - gini
	 numTrees - 20



RandomForestClassificationModel: uid=RandomForestClassifier_56434e717c68, numTrees=20, numClasses=2, numFeatures=2

In [34]:
rf_pca_best_pred = best_rf_pca.transform(test_pca)
rf_pca_best_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC").evaluate(rf_pca_best_pred)
rf_pca_best_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderPR").evaluate(rf_pca_best_pred)

print(f"Test area under ROC for best model: {rf_pca_best_roc}\nTest area under PR for best model: {rf_pca_best_pr}")

Test area under ROC for best model: 0.5364460892228033
Test area under PR for best model: 0.5459113602831631


## Model 2

The Factorization Machines algorithm is a supervised learning algorithm which is an extension of a linear model designed to capture interactions between features within high dimensional sparse datasets economically.

In [35]:
# Train second model
from pyspark.ml.classification import FMClassifier

fm_calssifier = FMClassifier()
fm_model = fm_calssifier.fit(train_df)

In [36]:
# Test second model
fm_predictions = fm_model.transform(test_df)
fm_predictions.show(10)

+--------------------+-----+--------------------+--------------------+----------+
|            features|label|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(481,[0,24,32,49,...|  1.0|[-1298.6000763708...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  0.0|[1008.81578525971...|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  1.0|[-714.11483595103...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  0.0|[699.421282569420...|[1.0,1.7587189958...|       0.0|
|(481,[0,24,32,49,...|  1.0|[-701.94022084224...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  1.0|[-868.63715751592...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  0.0|[1014.00353873349...|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  1.0|[-804.42337018110...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  1.0|[-1423.4812322554...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,4

In [37]:
# Evaluation
from pyspark.ml.evaluation import BinaryClassificationEvaluator

fm_evaluator_roc = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderROC")

fm_roc = fm_evaluator_roc.evaluate(fm_predictions)

fm_evaluator_pr = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderPR")

fm_pr = fm_evaluator_pr.evaluate(fm_predictions)

print(f"Test area under ROC: {fm_roc}\nTest area under PR: {fm_pr}")

Test area under ROC: 0.9994878437982756
Test area under PR: 0.9995326685422407


In [38]:
# Hyper-parameter optimization
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
print(f"Hyperparameters:\n\t regParam - {[0.0, 0.5]}\n\t initStd - {[0.01, 0.05]}\n\t factorSize - {[4, 8]}")

fm_grid = ParamGridBuilder()
fm_grid = fm_grid.addGrid(fm_model.regParam, [0.0, 0.5])\
                 .addGrid(fm_model.initStd, [0.01, 0.05])\
                 .addGrid(fm_model.factorSize, [4, 8])\
                 .build()

fm_cv = CrossValidator(estimator = fm_calssifier,
                    estimatorParamMaps = fm_grid,
                    evaluator = fm_evaluator_roc,
                    parallelism = 5,
                    numFolds=3)

fm_cvModel = fm_cv.fit(train_df)
model2 = fm_cvModel.bestModel

print(f"\nBest hyperparameters:\n\t initStd - {model2.getInitStd()}\n\t regParam - {model2.getRegParam()}\n\t factorSize - {model2.getFactorSize()}\n")

model2

Hyperparameters:
	 regParam - [0.0, 0.5]
	 initStd - [0.01, 0.05]
	 factorSize - [4, 8]

Best hyperparameters:
	 initStd - 0.05
	 regParam - 0.0
	 factorSize - 8



FMClassificationModel: uid=FMClassifier_7272abf3f5ed, numClasses=2, numFeatures=481, factorSize=8, fitLinear=true, fitIntercept=true

In [39]:
model2.write().overwrite().save("/user/team4/project/models/model2")
run("hdfs dfs -get /user/team4/project/models/model2 ~/project/bigdata-project/models/model2")

''

In [40]:
fm_predictions_grid = model2.transform(test_df)
fm_predictions_grid.show(10)

+--------------------+-----+--------------------+--------------------+----------+
|            features|label|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(481,[0,24,32,49,...|  0.0|[560.837709323785...|[1.0,2.6994643562...|       0.0|
|(481,[0,24,32,49,...|  0.0|[804.004305257378...|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  1.0|[-1527.6527217253...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  0.0|[713.892874267496...|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  0.0|[911.166506476949...|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  0.0|[859.857216061721...|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  0.0|[841.292959060188...|           [1.0,0.0]|       0.0|
|(481,[0,24,32,49,...|  1.0|[-1950.7360331765...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,49,...|  1.0|[-2470.7286125042...|           [0.0,1.0]|       1.0|
|(481,[0,24,32,4

In [41]:
fm_predictions_grid.select("label", "prediction")\
    .coalesce(1)\
    .write\
    .mode("overwrite")\
    .format("csv")\
    .option("sep", ",")\
    .option("header","true")\
    .save("/user/team4/project/output/model2_predictions")
run("hdfs dfs -cat /user/team4/project/output/model2_predictions/*.csv > ~/project/bigdata-project/output/model2_predictions.csv")

''

In [42]:
fm_evaluator_best_roc = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderROC")

fm_best_roc = fm_evaluator_best_roc.evaluate(fm_predictions_grid)

fm_evaluator_best_pr = BinaryClassificationEvaluator()\
  .setLabelCol("label")\
  .setRawPredictionCol("prediction")\
  .setMetricName("areaUnderPR")

fm_best_pr = fm_evaluator_best_pr.evaluate(fm_predictions_grid)

print(f"Test area under ROC for best model: {fm_best_roc}\nTest area under PR for best model: {fm_best_pr}")

Test area under ROC for best model: 0.9996983600028968
Test area under PR for best model: 0.9994661360264379


In [43]:
print(f"Test area under ROC for first model: {fm_roc}\nTest area under PR for first model: {fm_pr}")
print(f"Test area under ROC for best model: {fm_best_roc}\nTest area under PR for best model: {fm_best_pr}")
print(f"\nTest area under ROC increased by {fm_best_roc - fm_roc} with optimization\nTest area under PR increased " + 
      f"by {fm_best_pr - fm_pr} with optimization")

Test area under ROC for first model: 0.9994878437982756
Test area under PR for first model: 0.9995326685422407
Test area under ROC for best model: 0.9996983600028968
Test area under PR for best model: 0.9994661360264379

Test area under ROC increased by 0.0002105162046212028 with optimization
Test area under PR increased by -6.65325158027974e-05 with optimization


### Factorization Machines on PCA

In [44]:
# FM on PCA (basic)
fm_pca = FMClassifier(labelCol="label", featuresCol="features")
fm_pca_model = fm_pca.fit(train_pca)

fm_pca_pred = fm_pca_model.transform(test_pca)
fm_pca_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC").evaluate(fm_pca_pred)
fm_pca_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderPR").evaluate(fm_pca_pred)

print(f"Test area under ROC: {fm_pca_roc}\nTest area under PR: {fm_pca_pr}")

Test area under ROC: 0.5032153552750361
Test area under PR: 0.5038402988406434


In [45]:
# Tuning FM on PCA
fm_pca_grid = ParamGridBuilder() \
    .addGrid(fm_pca_model.regParam, [0.0, 0.5]) \
    .addGrid(fm_pca_model.initStd, [0.01, 0.05]) \
    .addGrid(fm_pca_model.factorSize, [4, 8]) \
    .build()

fm_pca_cv = CrossValidator(estimator=FMClassifier(labelCol="label", featuresCol="features"),
                           estimatorParamMaps=fm_pca_grid,
                           evaluator=BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC"),
                           parallelism=5, numFolds=3)

fm_pca_cvModel = fm_pca_cv.fit(train_pca)
best_fm_pca = fm_pca_cvModel.bestModel

print(f"\nBest hyperparameters:\n\t initStd - {best_fm_pca.getInitStd()}\n\t regParam - {best_fm_pca.getRegParam()}\n\t factorSize - {best_fm_pca.getFactorSize()}\n")

best_fm_pca


Best hyperparameters:
	 initStd - 0.01
	 regParam - 0.0
	 factorSize - 8



FMClassificationModel: uid=FMClassifier_cb087aec256d, numClasses=2, numFeatures=2, factorSize=8, fitLinear=true, fitIntercept=true

In [46]:
fm_pca_best_pred = best_fm_pca.transform(test_pca)
fm_pca_best_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC").evaluate(fm_pca_best_pred)
fm_pca_best_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderPR").evaluate(fm_pca_best_pred)

print(f"Test area under ROC for best model: {fm_pca_best_roc}\nTest area under PR for best model: {fm_pca_best_pr}")

Test area under ROC for best model: 0.50102374968801
Test area under PR for best model: 0.4998730408907802


## Model 3 (Neural Network - small 2‑layer MLP)

In [47]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

input_size = len(df_proj.select("features").first()[0])

# Basic MLP model without PCA
mlp_classifier = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features",
                                                layers=[input_size, 64, 2],
                                                maxIter=100, stepSize=0.03, blockSize=128, seed=42)
mlp_model = mlp_classifier.fit(train_df)

In [48]:
# Evaluate basic MLP
mlp_pred = mlp_model.transform(test_df)
mlp_evaluator_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC")
mlp_evaluator_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderPR")
mlp_roc = mlp_evaluator_roc.evaluate(mlp_pred)
mlp_pr = mlp_evaluator_pr.evaluate(mlp_pred)

print(f"Test area under ROC: {mlp_roc}\nTest area under PR: {mlp_pr}")

Test area under ROC: 0.9999547342360142
Test area under PR: 0.9999547886369186


In [49]:
# Hyperparameter tuning
print(f"Hyperparameters:\n\t stepSize - {[0.01, 0.03]}\n\t blockSize - {[128, 256]}\n\t hidden_layer_size - {[32, 64]}")

mlp_base = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features",
                                          maxIter=100, seed=42)

mlp_grid = ParamGridBuilder() \
    .addGrid(mlp_base.stepSize, [0.01, 0.03]) \
    .addGrid(mlp_base.blockSize, [128, 256]) \
    .addGrid(mlp_base.layers, [[input_size, 32, 2], [input_size, 64, 2]]) \
    .build()

mlp_cv = CrossValidator(estimator=mlp_base,
                        estimatorParamMaps=mlp_grid,
                        evaluator=mlp_evaluator_roc,
                        parallelism=5,
                        numFolds=3)
mlp_cvModel = mlp_cv.fit(train_df)
model3 = mlp_cvModel.bestModel

print(f"\nBest hyperparameters:\n\t stepSize - {model3.getStepSize()}\n\t blockSize - {model3.getBlockSize()}\n\t layers - {model3.getLayers()}\n")

model3

Hyperparameters:
	 stepSize - [0.01, 0.03]
	 blockSize - [128, 256]
	 hidden_layer_size - [32, 64]

Best hyperparameters:
	 stepSize - 0.01
	 blockSize - 256
	 layers - [481, 32, 2]



MultilayerPerceptronClassificationModel: uid=MultilayerPerceptronClassifier_f762f8bef872, numLayers=3, numClasses=2, numFeatures=481

In [50]:
# Evaluate best MLP
mlp_best_pred = model3.transform(test_df)
mlp_best_roc = mlp_evaluator_roc.evaluate(mlp_best_pred)
mlp_best_pr = mlp_evaluator_pr.evaluate(mlp_best_pred)

print(f"Test area under ROC for best model: {mlp_best_roc}\nTest area under PR for best model: {mlp_best_pr}")

Test area under ROC for best model: 1.0
Test area under PR for best model: 0.999992441098619


In [51]:
# Save model
model3.write().overwrite().save("/user/team4/project/models/model3")
run("hdfs dfs -get /user/team4/project/models/model3 ~/project/bigdata-project/models/model3")

''

### MLP on PCA

In [52]:
input_size_pca = len(train_pca.select("features").first()[0])
mlp_pca_base = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features",
                                              layers=[input_size_pca, 64, 2],
                                              maxIter=100, stepSize=0.03, blockSize=128, seed=42)
mlp_pca_model = mlp_pca_base.fit(train_pca)

In [53]:
mlp_pca_pred = mlp_pca_model.transform(test_pca)
mlp_pca_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC").evaluate(mlp_pca_pred)
mlp_pca_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderPR").evaluate(mlp_pca_pred)

print(f"Test area under ROC: {mlp_pca_roc}\nTest area under PR: {mlp_pca_pr}")

Test area under ROC: 0.5171783686933125
Test area under PR: 0.5118780663511112


In [56]:
# Tuning MLP on PCA
mlp_pca_grid = ParamGridBuilder() \
    .addGrid(mlp_pca_base.stepSize, [0.01, 0.03]) \
    .addGrid(mlp_pca_base.blockSize, [128, 256]) \
    .addGrid(mlp_pca_base.layers, [[input_size_pca, 32, 2], [input_size_pca, 64, 2]]) \
    .build()

mlp_pca_cv = CrossValidator(estimator=mlp_pca_base,
                            estimatorParamMaps=mlp_pca_grid,
                            evaluator=BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC"),
                            parallelism=5, numFolds=3)

mlp_pca_cvModel = mlp_pca_cv.fit(train_pca)
best_mlp_pca = mlp_pca_cvModel.bestModel

print(f"\nBest hyperparameters:\n\t stepSize - {best_mlp_pca.getStepSize()}\n\t blockSize - {best_mlp_pca.getBlockSize()}\n\t layers - {best_mlp_pca.getLayers()}\n")

best_mlp_pca


Best hyperparameters:
	 stepSize - 0.01
	 blockSize - 128
	 layers - [2, 64, 2]



MultilayerPerceptronClassificationModel: uid=MultilayerPerceptronClassifier_8d309425b6b7, numLayers=3, numClasses=2, numFeatures=2

In [57]:
mlp_pca_best_pred = best_mlp_pca.transform(test_pca)
mlp_pca_best_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC").evaluate(mlp_pca_best_pred)
mlp_pca_best_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderPR").evaluate(mlp_pca_best_pred)

print(f"Test area under ROC for best model: {mlp_pca_best_roc}\nTest area under PR for best model: {mlp_pca_best_pr}")

Test area under ROC for best model: 0.5192696159824364
Test area under PR for best model: 0.5164005221470849


## Comparison

In [58]:
# Create data frame to report evaluation metrics for all models (non-PCA and PCA)
# Models without PCA have numFeatures=481,
# and models with data after PCA have numFeatures=2
models = [
    [str(model1), rf_best_roc, rf_best_pr],
    [str(model2), fm_best_roc, fm_best_pr],
    [str(model3), mlp_best_roc, mlp_best_pr],
    [str(best_rf_pca), rf_pca_best_roc, rf_pca_best_pr],
    [str(best_fm_pca), fm_pca_best_roc, fm_pca_best_pr],
    [str(best_mlp_pca), mlp_pca_best_roc, mlp_pca_best_pr]
]

result_df = spark.createDataFrame(models, ["model", "Area under ROC", "Area under PR"])
result_df.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------+------------------+------------------+
|model                                                                                                                               |Area under ROC    |Area under PR     |
+------------------------------------------------------------------------------------------------------------------------------------+------------------+------------------+
|RandomForestClassificationModel: uid=RandomForestClassifier_652c75c2e8b0, numTrees=10, numClasses=2, numFeatures=481                |0.9998649743451256|0.9999550792426315|
|FMClassificationModel: uid=FMClassifier_7272abf3f5ed, numClasses=2, numFeatures=481, factorSize=8, fitLinear=true, fitIntercept=true|0.9996983600028968|0.9994661360264379|
|MultilayerPerceptronClassificationModel: uid=MultilayerPerceptronClassifier_f762f8bef872, numLayers=3, numClasses=2, numFeatures=481|1

In [59]:
# Save it to HDFS
result_df.coalesce(1)\
    .write\
    .mode("overwrite")\
    .format("csv")\
    .option("sep", ";")\
    .option("header","true")\
    .save("/user/team4/project/output/evaluation")
run("hdfs dfs -cat /user/team4/project/output/evaluation/*.csv > ~/project/bigdata-project/output/evaluation.csv")

''

## Store additional results

### Get best hyperparameters

In [60]:
# Create data frame to report best hyperparameters for all models (non-PCA and PCA)
hypoparams = [
    [str(model1), f'maxDepth = {model1.getMaxDepth()}', f'impurity = {model1.getImpurity()}', f'numTrees = {model1.getNumTrees}'],
    [str(model2), f'initStd = {model2.getInitStd()}', f'regParam = {model2.getRegParam()}', f'factorSize = {model2.getFactorSize()}'],
    [str(model3), f'stepSize = {model3.getStepSize()}', f'blockSize = {model3.getBlockSize()}', f'layers = {model3.getLayers()}'],
    [str(best_rf_pca), f'maxDepth = {best_rf_pca.getMaxDepth()}', f'impurity = {best_rf_pca.getImpurity()}', f'numTrees = {best_rf_pca.getNumTrees}'],
    [str(best_fm_pca), f'initStd = {best_fm_pca.getInitStd()}', f'regParam = {best_fm_pca.getRegParam()}', f'factorSize = {best_fm_pca.getFactorSize()}'],
    [str(best_mlp_pca), f'stepSize = {best_mlp_pca.getStepSize()}', f'blockSize = {best_mlp_pca.getBlockSize()}', f'layers = {best_mlp_pca.getLayers()}']
]

hypo_df = spark.createDataFrame(hypoparams, ["model", "Parameter 1", "Parameter 2", "Parameter 3"])
hypo_df.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------+---------------+---------------+---------------------+
|model                                                                                                                               |Parameter 1    |Parameter 2    |Parameter 3          |
+------------------------------------------------------------------------------------------------------------------------------------+---------------+---------------+---------------------+
|RandomForestClassificationModel: uid=RandomForestClassifier_652c75c2e8b0, numTrees=10, numClasses=2, numFeatures=481                |maxDepth = 10  |impurity = gini|numTrees = 10        |
|FMClassificationModel: uid=FMClassifier_7272abf3f5ed, numClasses=2, numFeatures=481, factorSize=8, fitLinear=true, fitIntercept=true|initStd = 0.05 |regParam = 0.0 |factorSize = 8       |
|MultilayerPerceptronClassificationModel: uid=Multilaye

In [61]:
# Save it to HDFS
hypo_df.coalesce(1)\
    .write\
    .mode("overwrite")\
    .format("csv")\
    .option("sep", ";")\
    .option("header","true")\
    .save("/user/team4/project/output/hyperparameters")
run("hdfs dfs -cat /user/team4/project/output/hyperparameters/*.csv > ~/project/bigdata-project/output/hyperparameters.csv")

''

### Get result of optimization

In [62]:
# Create data frame to report initial and optimized metrics (no increase column)
optim_models = [
    [str(model1), rf_roc, rf_best_roc, rf_pr, rf_best_pr],
    [str(model2), fm_roc, fm_best_roc, fm_pr, fm_best_pr],
    [str(model3), mlp_roc, mlp_best_roc, mlp_pr, mlp_best_pr],
    [str(best_rf_pca), rf_pca_roc, rf_pca_best_roc, rf_pca_pr, rf_pca_best_pr],
    [str(best_fm_pca), fm_pca_roc, fm_pca_best_roc, fm_pca_pr, fm_pca_best_pr],
    [str(best_mlp_pca), mlp_pca_roc, mlp_pca_best_roc, mlp_pca_pr, mlp_pca_best_pr]
]

optim_df = spark.createDataFrame(optim_models, ["model", "Initial area under ROC", "Optimized area under ROC",
                                                "Initial area under PR", "Optimized area under PR"])
optim_df.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------+----------------------+------------------------+---------------------+-----------------------+
|model                                                                                                                               |Initial area under ROC|Optimized area under ROC|Initial area under PR|Optimized area under PR|
+------------------------------------------------------------------------------------------------------------------------------------+----------------------+------------------------+---------------------+-----------------------+
|RandomForestClassificationModel: uid=RandomForestClassifier_652c75c2e8b0, numTrees=10, numClasses=2, numFeatures=481                |0.9999100746380505    |0.9998649743451256      |0.9998574216520281   |0.9999550792426315     |
|FMClassificationModel: uid=FMClassifier_7272abf3f5ed, numClasses=2, numFeatures=481

In [63]:
# Save it to HDFS
optim_df.coalesce(1)\
    .write\
    .mode("overwrite")\
    .format("csv")\
    .option("sep", ";")\
    .option("header","true")\
    .save("/user/team4/project/output/optimization")

run("hdfs dfs -cat /user/team4/project/output/optimization/*.csv > ~/project/bigdata-project/output/optimization.csv")

''

### Predict sample on all models

In [64]:
# Get random sample and predict on all three models
prediction_sample = df_labeled.sample(fraction=10/df_labeled.count(), seed=123)
models_pred = model3.transform(model2.transform(model1.transform(prediction_sample)\
                 .withColumnRenamed("prediction", "model1_prediction")\
                 .withColumnRenamed("rawPrediction", "model1_rawPrediction")\
                 .withColumnRenamed("probability", "model1_probability"))\
                 .withColumnRenamed("prediction", "model2_prediction")\
                 .withColumnRenamed("rawPrediction", "model2_rawPrediction")\
                 .withColumnRenamed("probability", "model2_probability"))\
                 .withColumnRenamed("prediction", "model3_prediction")

In [65]:
prediction_result = models_pred.select(*(numerical + cyclical +\
                                               time_features + date_features +\
                                               boolean_features + categorical_as_cont +\
                                               categorical_ohe + ['label', "model1_prediction", "model2_prediction", "model3_prediction"]))

In [66]:
# Save it to HDFS
prediction_result.coalesce(1)\
    .write\
    .mode("overwrite")\
    .format("csv")\
    .option("sep", ";")\
    .option("header","true")\
    .save("/user/team4/project/output/prediction_samples")

run("hdfs dfs -cat /user/team4/project/output/prediction_samples/*.csv > ~/project/bigdata-project/output/prediction_samples.csv")

''

## Model Interpretation

### Random Forest Feature Importance

In [73]:
from pyspark.ml.feature import OneHotEncoderModel
from collections import defaultdict

# Extract one‑hot vector sizes from the trained pipeline
ohe_models = [s for s in features_pipeline_model.stages if isinstance(s, OneHotEncoderModel)]
ohe_sizes = {}
for ohe in ohe_models:
    out_col = ohe.getOutputCol()          # e.g. "airline_idx_enc"
    cat_size = ohe.categorySizes[0]       # total number of categories
    vec_size = cat_size - 1 if ohe.getDropLast() else cat_size
    ohe_sizes[out_col] = vec_size

In [75]:
# Build a list component_origin: for each position in the feature vector
# store the original column name (before StringIndexer/OneHotEncoder)
component_origin = []

# One‑hot encoded features (categorical_ohe)
for i, f_idx in enumerate(features_idx):
    enc_col = f_idx + "_enc"
    size = ohe_sizes.get(enc_col, 1)
    orig_name = categorical_ohe[i]        # original column before indexing
    component_origin.extend([orig_name] * size)

# Categorical features treated as continuous (categorical_as_cont)
for i, f_idx in enumerate(features_cat_idx):
    component_origin.append(categorical_as_cont[i])

# Numerical features
component_origin.extend(numerical)

# Boolean features
component_origin.extend(boolean_features)

# Time features (8 components per time_feature)
for t in time_features:
    component_origin.extend([t] * 8)

# Date features (4 components per date_feature)
for d in date_features:
    component_origin.extend([d] * 4)

# Cyclical features (2 components sin/cos each)
for c in cyclical:
    component_origin.extend([c] * 2)

In [76]:
# Compute aggregated importance (absolute value) per original feature
rf_importance = model1.featureImportances.toArray()
importance_sum = defaultdict(float)
for name, imp in zip(component_origin, rf_importance):
    importance_sum[name] += float(abs(imp))

top_n = 10
sorted_importance = sorted(importance_sum.items(), key=lambda x: x[1], reverse=True)[:top_n]

In [77]:
# Create and save the result table
schema = StructType([
    StructField("feature", StringType(), True),
    StructField("importance", DoubleType(), True)
])
rf_imp_orig_df = spark.createDataFrame(sorted_importance, schema=schema)
rf_imp_orig_df.show(truncate=False)

+--------------------+--------------------+
|feature             |importance          |
+--------------------+--------------------+
|wheelsoff           |0.4628020074316001  |
|crsarrtime          |0.15776158531754964 |
|arrtime             |0.14391129676953793 |
|departuredelaygroups|0.08010868429709339 |
|deptime             |0.0578107361020138  |
|taxiout             |0.03244803148114123 |
|arrivaldelaygroups  |0.01611636562887592 |
|depdelayminutes     |0.014890176839621844|
|crsdeptime          |0.013839748955327575|
|wheelson            |0.007300008240948187|
+--------------------+--------------------+



In [78]:
rf_imp_orig_df.coalesce(1)\
    .write.mode("overwrite")\
    .format("csv")\
    .option("sep", ";")\
    .option("header", "true")\
    .save("/user/team4/project/output/rf_feature_importance_original")

run("hdfs dfs -cat /user/team4/project/output/rf_feature_importance_original/*.csv > ~/project/bigdata-project/output/rf_feature_importance_original.csv")

''

### Factorization Machines Interpretation (SHAP with mapInPandas)

In [106]:
# import sys, site
# !{sys.executable} -m pip install shap --quiet
# sys.path.insert(0, site.getusersitepackages())  # ~/.local/lib/pythonX.Y/site-packages

In [117]:
# Factorization Machines SHAP Interpretation
import shap
import numpy as np
import pandas as pd
from pyspark.ml.linalg import Vectors

In [129]:
sample_pd = test_df.select("features", "label").limit(20).toPandas()
X_bg = np.array([v.toArray() for v in sample_pd["features"]], dtype=np.float64)

In [130]:
# Prepare background data and prediction function
feature_names = [f"f{i}" for i in range(X_bg.shape[1])]
background = shap.sample(X_bg, min(20, len(X_bg)))  # plain numpy array

In [131]:
def fm_predict_proba_np(X):
    """Return probability of class 1 for the FM model (numpy -> numpy)."""
    # Convert each row to a dense vector and build a Spark DataFrame with correct type
    rows = [(Vectors.dense(x),) for x in X]
    spark_X = spark.createDataFrame(rows, schema=["features"])
    preds = model2.transform(spark_X)
    proba = np.array(preds.select("probability").rdd.map(lambda r: float(r[0][1])).collect())
    return proba

explainer_fm = shap.KernelExplainer(fm_predict_proba_np, background)
X_explain = X_bg[:20]   # first 20 instances of the background
shap_values_fm = explainer_fm.shap_values(X_explain, nsamples=500, silent=True)

mean_abs_fm = np.mean(np.abs(shap_values_fm), axis=0)

In [140]:
# Build and save the top-10 feature importance table
from collections import defaultdict

cumulative_fm = defaultdict(float)
for i, col_name in enumerate(component_origin):
    cumulative_fm[col_name] += float(mean_abs_fm[i])

# Sort by cumulative importance descending and take top_n
top_n = 10
sorted_fm = sorted(cumulative_fm.items(), key=lambda x: x[1], reverse=True)[:top_n]

In [141]:
schema_fm_agg = StructType([
    StructField("feature", StringType(), True),
    StructField("mean_abs_shap", DoubleType(), True)
])
fm_agg_df = spark.createDataFrame(sorted_fm, schema=schema_fm_agg)
fm_agg_df.show(truncate=False)

+----------+--------------------+
|feature   |mean_abs_shap       |
+----------+--------------------+
|wheelsoff |0.1153965803662678  |
|deptime   |0.06731991106660898 |
|arrdel15  |0.06268320347006015 |
|arrtime   |0.05373008749267147 |
|wheelson  |0.05324484338021619 |
|deptimeblk|0.030633462175360303|
|arrtimeblk|0.027873152538081057|
|crsarrtime|0.020746718256855454|
|crsdeptime|0.015146507394888414|
|flightdate|0.009110969222807666|
+----------+--------------------+



In [142]:
fm_agg_df.coalesce(1)\
    .write.mode("overwrite")\
    .format("csv")\
    .option("sep", ";")\
    .option("header", "true")\
    .save("/user/team4/project/output/fm_shap_importance")

run("hdfs dfs -cat /user/team4/project/output/fm_shap_importance/*.csv > ~/project/bigdata-project/output/fm_shap_importance.csv")

''

### Neural Network (MLP) Interpretation (SHAP with mapInPandas)

In [135]:
# Multilayer Perceptron SHAP Interpretation
import shap
import numpy as np
from pyspark.ml.linalg import Vectors

# Background and prediction function
feature_names = [f"f{i}" for i in range(X_bg.shape[1])]
background = shap.sample(X_bg, min(20, len(X_bg)))   # same as FM

In [136]:
def mlp_predict_proba_np(X):
    """Return probability of class 1 for the MLP model (numpy -> numpy)."""
    rows = [(Vectors.dense(x),) for x in X]
    spark_X = spark.createDataFrame(rows, schema=["features"])
    preds = model3.transform(spark_X)
    proba = np.array(preds.select("probability").rdd.map(lambda r: float(r[0][1])).collect())
    return proba

explainer_mlp = shap.KernelExplainer(mlp_predict_proba_np, background)
X_explain = X_bg[:20]
shap_values_mlp = explainer_mlp.shap_values(X_explain, nsamples=500, silent=True)

mean_abs_mlp = np.mean(np.abs(shap_values_mlp), axis=0)

In [143]:
# Build and save the top-10 feature importance table
from collections import defaultdict

# Aggregate mean absolute SHAP values per original feature name
cumulative_mlp = defaultdict(float)
for i, col_name in enumerate(component_origin):
    cumulative_mlp[col_name] += float(mean_abs_mlp[i])

# Sort and take top_n (renamed column "feature")
top_n = 10
sorted_mlp = sorted(cumulative_mlp.items(), key=lambda x: x[1], reverse=True)[:top_n]

In [144]:
schema_agg = StructType([
    StructField("feature", StringType(), True),
    StructField("mean_abs_shap", DoubleType(), True)
])
mlp_agg_df = spark.createDataFrame(sorted_mlp, schema=schema_agg)
mlp_agg_df.show(truncate=False)

+----------+--------------------+
|feature   |mean_abs_shap       |
+----------+--------------------+
|wheelsoff |0.14542240477662324 |
|wheelson  |0.07506740722007563 |
|deptime   |0.06209354478766384 |
|arrtime   |0.04793347082617584 |
|deptimeblk|0.03429804871330271 |
|arrtimeblk|0.025206501888772832|
|crsdeptime|0.024597178704191376|
|arrdel15  |0.015747940148804623|
|crsarrtime|0.013516415608742637|
|month     |0.0117894697179323  |
+----------+--------------------+



In [145]:
mlp_agg_df.coalesce(1)\
    .write.mode("overwrite")\
    .format("csv")\
    .option("sep", ";")\
    .option("header", "true")\
    .save("/user/team4/project/output/mlp_shap_importance")

run("hdfs dfs -cat /user/team4/project/output/mlp_shap_importance/*.csv > ~/project/bigdata-project/output/mlp_shap_importance.csv")

''

## Deployment Risk Analysis

### Risk 1: Timestamp Misalignment Risk

In [265]:
from pyspark.sql.functions import rand, when, col, unix_timestamp, from_unixtime, to_timestamp, expr
from pyspark.sql.types import IntegerType
import numpy as np

def corrupt_timestamps(df_sample, max_offset_minutes=120):
    # Take a small sample for speed
    raw_sample = df_sample.limit(2000)
    
    # Add a random offset (in seconds) to each timestamp column
    for ts_col in ['wheelsoff', 'arrtime', 'deptime']:
        # Generate random offset in seconds from -max_offset_minutes*60 to +max_offset_minutes*60
        offset_seconds = (rand() * 2 * max_offset_minutes * 60 - max_offset_minutes * 60).cast(IntegerType())
        
        # Apply offset only to 30% of rows (simulating a failure)
        raw_sample = raw_sample.withColumn(
            ts_col,
            when(
                rand() < 0.3,
                to_timestamp(from_unixtime(unix_timestamp(col(ts_col)) + offset_seconds))
            ).otherwise(col(ts_col))
        )
    
    # Re-apply the full feature extraction pipeline
    corrupted_feat = features_pipeline_model.transform(raw_sample)
    corrupted_feat = label_idx_model.transform(corrupted_feat)
    corrupted_feat = corrupted_feat.select('features', 'label')
    
    # Predict using the original Random Forest model
    preds = model1.transform(corrupted_feat)
    return preds

# Run the analysis
corrupted_preds = corrupt_timestamps(df_sampled, max_offset_minutes=120)
corrupted_preds.select('label', 'prediction', 'probability').show(10), ''

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  1.0|       1.0|[1.05042016806722...|
|  1.0|       1.0|           [0.0,1.0]|
|  1.0|       1.0|[2.13174163291409...|
|  0.0|       0.0|           [0.8,0.2]|
|  0.0|       0.0|[0.79991398342747...|
|  0.0|       0.0|[0.79991398342747...|
|  0.0|       0.0|           [0.8,0.2]|
|  0.0|       0.0|[0.71057388190463...|
|  0.0|       0.0|           [0.8,0.2]|
|  0.0|       0.0|           [0.8,0.2]|
+-----+----------+--------------------+
only showing top 10 rows



(None, '')

In [266]:
# Evaluate AUC-ROC on corrupted data
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='prediction', metricName='areaUnderROC')
auc_corrupted = evaluator.evaluate(corrupted_preds)
print(f"AUC-ROC after timestamp corruption: {auc_corrupted:.4f}")

AUC-ROC after timestamp corruption: 0.9988


### Risk 2: Unseen categorical entities

In [186]:
# Generate fake data with unseen airlines, origins, destinations, tail numbers
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DateType, TimestampType

num_per_class = 1000
np.random.seed(42)

# Sample templates for each class
sample_0 = df.filter(F.col("cancelled") == ).sample(True, 2.0, seed=1).limit(num_per_class)
sample_1 = df.filter(F.col("cancelled") == 1).sample(True, 3.0, seed=1).limit(num_per_class)

In [ ]:
# New categorical values never seen in training
new_airlines = [f"NEW_AIRLINE_{i}" for i in range(5)]
new_origins = [f"NEW_ORG_{i}" for i in range(3)]
new_dests = [f"NEW_DST_{i}" for i in range(3)]
new_tails = [f"N{str(i).zfill(5)}" for i in range(10)]

airlines_arr = F.array([F.lit(a) for a in new_airlines])
origins_arr = F.array([F.lit(o) for o in new_origins])
dests_arr = F.array([F.lit(d) for d in new_dests])
tails_arr = F.array([F.lit(t) for t in new_tails])

In [ ]:
# Replace categorical columns with unseen values
def replace_cats(data):
    return (data
        .withColumn("airline", F.element_at(airlines_arr, (F.rand(seed=1)*5).cast("int")+1))
        .withColumn("origin", F.element_at(origins_arr,  (F.rand(seed=2)*3).cast("int")+1))
        .withColumn("dest", F.element_at(dests_arr,    (F.rand(seed=3)*3).cast("int")+1))
        .withColumn("tail_number", F.element_at(tails_arr,    (F.rand(seed=4)*10).cast("int")+1))
        .withColumn("origincityname", F.lit("UnknownCity"))
        .withColumn("destcityname", F.lit("UnknownCity"))
    )

fake_0 = replace_cats(sample_0)
fake_1 = replace_cats(sample_1)
fake_all = fake_0.union(fake_1)

In [188]:
# Cast boolean columns to integer
boolean_features = ['diverted', 'depdel15', 'arrdel15']
LABEL = 'cancelled'
for b in boolean_features + [LABEL]:
    fake_all = fake_all.withColumn(b, F.col(b).cast('integer'))

    
# Encode time & date features 
for t in time_features:
    fake_all = fake_all.withColumn(t+'_hour', F.hour(t))
    fake_all = fake_all.withColumn(t+'_minute', F.minute(t))
    fake_all = fake_all.withColumn(t+'_day', F.dayofmonth(t))
    fake_all = fake_all.withColumn(t+'_month', F.month(t))

for d in date_features:
    fake_all = fake_all.withColumn(d+'_day', F.dayofmonth(t))
    fake_all = fake_all.withColumn(d+'_month', F.month(t))

# Apply trained pipelines and predict
df_fake_rf1 = features_pipeline_model.transform(fake_all)
df_fake_rf1 = label_idx_model.transform(df_fake_rf1)
preds_rf_risk1_rf = model1.transform(df_fake_rf1)

In [209]:
# Compute average probabilities per class
probs_rdd_rf = preds_rf_risk1_rf.select("label", "probability").rdd.map(lambda r: (float(r[0]), float(r[1][0]), float(r[1][1])))
probs_class0 = probs_rdd_rf.filter(lambda x: x[0] == 0.0).map(lambda x: x[1]).collect()
probs_class1 = probs_rdd_rf.filter(lambda x: x[0] == 1.0).map(lambda x: x[2]).collect()

def stats(vals):
    return (float(np.mean(vals)), float(np.std(vals))) if vals else (0.0, 0.0)

mean0, std0 = stats(probs_class0)
mean1, std1 = stats(probs_class1)
mean_all, std_all = stats(probs_class0 + probs_class1)

print(f"Class 0: mean={mean0:.4f}, std={std0:.4f}")
print(f"Class 1: mean={mean1:.4f}, std={std1:.4f}")
print(f"Overall: mean={mean_all:.4f}, std={std_all:.4f}")


Class 0: mean=0.9959, std=0.0258
Class 1: mean=0.9968, std=0.0121
Overall: mean=0.9964, std=0.0202


In [211]:
# Save summary statistics
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
schema = StructType([
    StructField("class", StringType()),
    StructField("mean_prob", DoubleType()),
    StructField("std_prob", DoubleType())
])
spark.createDataFrame([
    ("0", mean0, std0),
    ("1", mean1, std1),
    ("total", mean_all, std_all)
], schema).coalesce(1).write.mode("overwrite").format("csv") \
    .option("sep", ";").option("header", "true") \
    .save("/user/team4/project/output/risk1_rf_summary")

run("hdfs dfs -cat /user/team4/project/output/risk1_rf_summary/*.csv > ~/project/bigdata-project/output/risk1_rf_summary.csv")


''

### Risk 4: Operational Schedule Drift

In [ ]:
from pyspark.ml.classification import MultilayerPerceptronClassificationModel
from pyspark.sql.functions import concat, lit, date_format, to_timestamp, unix_timestamp, from_unixtime, hour, minute, col
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Choose the target time block (exists in training)
target_block = '0600-0659'

# Create a balanced sample (50% cancelled, 50% not cancelled)
cancelled_df = df_sampled.filter(col('cancelled') == 1).limit(1000)
not_cancelled_df = df_sampled.filter(col('cancelled') == 0).limit(1000)
balanced_sample = cancelled_df.union(not_cancelled_df)

# Keep only rows that originally do NOT belong to the target block
other_flights = balanced_sample.filter(col('deptimeblk') != target_block)

In [ ]:
# Function to shift timestamps so that deptime falls into the target block
def shift_to_target_block(df, target_hour=6, target_minute=0):
    """
    Shift all timestamp columns so that the departure time falls into the target hour:minute.
    The shift is applied uniformly to all timestamp columns.
    """
    # Build target datetime string using concat (not +)
    target_datetime_str = concat(date_format('flightdate', 'yyyy-MM-dd'), lit(f' {target_hour:02d}:{target_minute:02d}:00'))
    target_deptime = to_timestamp(target_datetime_str, 'yyyy-MM-dd HH:mm:ss')
    
    # Compute shift in seconds
    shift_seconds = unix_timestamp(target_deptime) - unix_timestamp(col('deptime'))
    
    # Shift all timestamp columns by the same amount
    timestamp_cols = ['deptime', 'crsdeptime', 'arrtime', 'crsarrtime', 'wheelsoff', 'wheelson']
    for ts_col in timestamp_cols:
        df = df.withColumn(ts_col, to_timestamp(from_unixtime(unix_timestamp(col(ts_col)) + shift_seconds)))
    return df

# Apply the shift
shifted_flights = shift_to_target_block(other_flights, target_hour=6, target_minute=0)

# Set deptimeblk to the target block (the pipeline does not recompute it)
shifted_flights = shifted_flights.withColumn('deptimeblk', lit(target_block))

In [283]:
# Function to compute average predicted probability of cancellation (class 1)
def get_avg_prob(df, model):
    feats = features_pipeline_model.transform(df)
    feats = label_idx_model.transform(feats)
    feats = feats.select('features', 'label')
    preds = model.transform(feats)
    get_prob_class1 = F.udf(lambda vec: float(vec[1]) if vec and len(vec) > 1 else 0.0, DoubleType())
    avg = preds.select(F.avg(get_prob_class1(F.col("probability"))).alias("avg_prob")).collect()[0][0]
    return avg

# Average probability for shifted flights
avg_prob_shifted = get_avg_prob(shifted_flights, model3)
print(f"Average predicted non-cancellation probability for shifted flights (in block {target_block}): {avg_prob_shifted:.4f}")

# Comparison with genuine flights in the target block (if any)
genuine_flights = balanced_sample.filter(col('deptimeblk') == target_block)
avg_prob_genuine = get_avg_prob(genuine_flights, model3)
print(f"Average predicted non-cancellation probability for genuine flights in block {target_block}: {avg_prob_genuine:.4f}")
print(f"Difference (shifted - genuine): {avg_prob_shifted - avg_prob_genuine:.4f}")

Average predicted non-cancellation probability for shifted flights (in block 0600-0659): 0.5008
Average predicted non-cancellation probability for genuine flights in block 0600-0659: 0.5567
Difference (shifted - genuine): -0.0559


## Stop Spark

In [46]:
spark.stop()